In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Getting totally carried away...
## Multiple linear regression

In [ ]:
# --- Generate synthetic dataset ---
np.random.seed(42)
n = 150

temperature = np.linspace(55, 100, n) + np.random.normal(0, 1.5, n)
humidity    = np.random.uniform(20, 90, n)
day_of_week = np.random.randint(1, 8, n)   # 1 = Monday, 7 = Sunday

# Sales formula: temperature helps a lot, humidity hurts a little,
# weekends (higher day number) boost sales modestly
sales = (
    8.5 * temperature
    - 1.2 * humidity
    + 4.0 * day_of_week
    - 180
    + np.random.normal(0, 20, n)
)

df = pd.DataFrame({
    'temperature_f': np.round(temperature, 1),
    'humidity':      np.round(humidity, 1),
    'day_of_week':   day_of_week,
    'sales_dollars': np.round(sales, 2)
})

print('Dataset preview:')
print(df.head(10))
print(f'\nDataset shape: {df.shape}')

In [ ]:
# --- Step 1: Define features and target ---
feature_cols = ['temperature_f', 'humidity', 'day_of_week']
X = df[feature_cols]
y = df['sales_dollars']

# --- Step 2: Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Step 3: Train the model ---
model = LinearRegression()
model.fit(X_train, y_train)

# --- Step 4: Predict ---
y_pred = model.predict(X_test)

In [ ]:
# --- Step 5: Evaluate ---
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f'\nMean Absolute Error: {mae:.2f}')
print(f'R² Score:            {r2:.4f}')
print(f'\nIntercept: {model.intercept_:.4f}')
print('\nLearned coefficients:')
for name, coef in zip(feature_cols, model.coef_):
    print(f'  {name:<18} {coef:.4f}')

In [ ]:
# --- Step 6: Visualize ---
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# --- Plot 1: Coefficient bar chart ---
colors = ['steelblue' if c > 0 else 'tomato' for c in model.coef_]

axes[0].bar(feature_cols, model.coef_, color=colors, width=.5)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Coefficient Value')
axes[0].set_title('Feature Coefficients\n(positive = boosts sales, negative = reduces sales)')

for i, (val, name) in enumerate(zip(model.coef_, feature_cols)):
    axes[0].text(i, val + (0.2 if val > 0 else -0.3),
                 f'{val:.2f}', ha='center', fontsize=10)

# --- Plot 2: Actual vs. predicted ---
axes[1].scatter(y_test, y_pred, color='steelblue', alpha=0.7, zorder=3)

combined_min = min(y_test.min(), y_pred.min())
combined_max = max(y_test.max(), y_pred.max())
axes[1].plot([combined_min, combined_max],
             [combined_min, combined_max],
             color='tomato', linewidth=1.5, linestyle='--', label='Perfect prediction')

axes[1].set_xlabel('Actual Sales ($)')
axes[1].set_ylabel('Predicted Sales ($)')
axes[1].set_title('Actual vs. Predicted Sales\n(closer to the line = better)')
axes[1].legend()

# --- Plot 3: Residuals ---
residuals = y_test - y_pred

axes[2].scatter(y_pred, residuals, color='seagreen', alpha=0.7, zorder=3)
axes[2].axhline(0, color='tomato', linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Predicted Sales ($)')
axes[2].set_ylabel('Residual ($)')
axes[2].set_title('Residual Plot\n(should look like random scatter around zero)')

plt.suptitle('Multiple Linear Regression: Ice Cream Sales', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()